## Track B: Multicultural Visual Reasoning

### 1. Environment Setup & OpenSearch Initialization

In [2]:
import os
import ast
import io
import base64
import pandas as pd
import numpy as np
from dotenv import load_dotenv
from opensearchpy import OpenSearch, helpers
from sentence_transformers import SentenceTransformer
from datasets import load_dataset, Dataset
from openai import OpenAI
from PIL import Image

load_dotenv()

OPENSEARCH_USER = os.getenv("OPENSEARCH_USER")
OPENSEARCH_PASSWORD = os.getenv("OPENSEARCH_PASSWORD")
OPENSEARCH_HOST = os.getenv("OPENSEARCH_HOST")
OPENSEARCH_PORT = os.getenv("OPENSEARCH_PORT")

BASE_URL = os.getenv("BASE_URL") or "https://api.novasearch.org/gemma4/v1"
API_KEY = os.getenv("API_KEY") or "nova-vl"
MODEL = os.getenv("MODEL") or "google/gemma-4-31b-it"

# Define target Track B Indices
cvqa_index_name = f"{OPENSEARCH_USER}_cvqa_project"
wiki_cache_index = f"{OPENSEARCH_USER}_wiki_cache"

# Initialize OpenSearch Client
client = OpenSearch(
    hosts=[{'host': OPENSEARCH_HOST, 'port': OPENSEARCH_PORT}],
    http_compress=True, 
    http_auth=(OPENSEARCH_USER, OPENSEARCH_PASSWORD),
    use_ssl=True,
    url_prefix='opensearch_v3',
    verify_certs=False,
    ssl_assert_hostname=False,
    ssl_show_warn=False
)

# Initialize OpenAI server client for Gemma-4-31B with verified fallbacks
openai_client = OpenAI(base_url=BASE_URL, api_key=API_KEY)

# Initialize embedding models matching your Phase 2 vector fields
print("Loading embedding models...")
#sbert_model = SentenceTransformer('all-mpnet-base-v2')       # 768 dim
#bge_model = SentenceTransformer('BAAI/bge-small-en-v1.5')     # 384 dim
#clip_model = SentenceTransformer('clip-ViT-B-32')             # 512 dim
bge_model= SentenceTransformer('BAAI/bge-m3') # 1024 dim, multilingual
print("Models loaded successfully.")

Loading embedding models...


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Models loaded successfully.


### 2. Dataset Loading and Stratified Held-out Split

In [3]:
from datasets import Value
print("Loading afaji/cvqa dataset from Hugging Face...")
cvqa_ds = load_dataset("afaji/cvqa", split="test")

def parse_subset_metadata(example):
    try:
        # Extract Language and Country safely from the Subset tuple string
        subset_tuple = ast.literal_eval(example['Subset'])
        example['language'] = subset_tuple[0]
        example['country'] = subset_tuple[1]
    except:
        example['language'] = "Unknown"
        example['country'] = "Unknown"
    return example

# Map metadata and filter for target evaluation languages
cvqa_ds = cvqa_ds.map(parse_subset_metadata)
df_full = cvqa_ds.to_pandas()
print("Full dataset language distribution:")
print(df_full['language'].value_counts())
target_languages = ["English", "Portuguese", "Egyptian_Arabic"]
cvqa_filtered = cvqa_ds.filter(lambda x: x['language'] in target_languages)

# Convert to pandas to sample 1,000 rows proportionally
df_cvqa = cvqa_filtered.to_pandas()
sampled_df = df_cvqa.groupby('language', group_keys=False)[df_cvqa.columns].apply(
    lambda x: x.sample(min(len(x), 334), random_state=42)
).reset_index(drop=True)

# Convert back to Dataset and enforce exactly 1000 items
working_dataset = Dataset.from_pandas(sampled_df, preserve_index=False).shuffle(seed=42)
if len(working_dataset) > 1000:
    working_dataset = working_dataset.select(range(1000))

#encode for stratified splitting
working_dataset = working_dataset.class_encode_column("language")
label_names = working_dataset.features['language'].names

# Create stratified Train (Retrieval Corpus) and Held-Out Test Set
split_ds = working_dataset.train_test_split(test_size=0.2, stratify_by_column='language', seed=42)

#decode language back to string
retrieval_corpus = split_ds['train'].map(lambda x: {"language": label_names[x['language']]})
retrieval_corpus = retrieval_corpus.cast_column("language", Value("string"))

test_set = split_ds['test'].map(lambda x: {"language": label_names[x['language']]})
test_set = test_set.cast_column("language", Value("string"))

print(f"Dataset Split complete: Retrieval Split size = {len(retrieval_corpus)}, Blind Test Split size = {len(test_set)}")
print("Test set language distribution:")
print(pd.Series([row['language'] for row in test_set]).value_counts())
print("Test set country distribution:")
print(pd.Series([row['country'] for row in test_set]).value_counts())
print("Train set language distribution:")
print(pd.Series([row['language'] for row in retrieval_corpus]).value_counts())
print("Train set country distribution:")
print(pd.Series([row['country'] for row in retrieval_corpus]).value_counts())

Loading afaji/cvqa dataset from Hugging Face...
Full dataset language distribution:
language
Spanish            2058
Chinese             523
Urdu                436
Indonesian          412
Breton              405
Bulgarian           371
Irish               326
Malay               315
Mongolian           312
Romanian            302
Norwegian           299
Javanese            297
Korean              290
Bengali             286
Portuguese          284
Swahili             273
Minangkabau         251
Kinyarwanda         235
Amharic             234
Sinhala             225
Tamil               214
Oromo               214
Filipino            203
Egyptian_Arabic     203
Japanese            203
Marathi             202
Hindi               201
Sundanese           200
Telugu              200
Igbo                200
Russian             200
Name: count, dtype: int64


Stringifying the column:   0%|          | 0/487 [00:00<?, ? examples/s]

Casting to class labels:   0%|          | 0/487 [00:00<?, ? examples/s]

Map:   0%|          | 0/389 [00:00<?, ? examples/s]

Casting the dataset:   0%|          | 0/389 [00:00<?, ? examples/s]

Map:   0%|          | 0/98 [00:00<?, ? examples/s]

Casting the dataset:   0%|          | 0/98 [00:00<?, ? examples/s]

Dataset Split complete: Retrieval Split size = 389, Blind Test Split size = 98
Test set language distribution:
1    57
0    41
Name: count, dtype: int64
Test set country distribution:
Brazil    57
Egypt     41
Name: count, dtype: int64
Train set language distribution:
1    227
0    162
Name: count, dtype: int64
Train set country distribution:
Brazil    227
Egypt     162
Name: count, dtype: int64


Indexing: Index the retrieval split (80%) of CVQA images, questions, and
answer options for your chosen languages in OpenSearch. Include language
and cultural region as metadata fields to support filtered retrieval.


In [4]:
print(retrieval_corpus.column_names)
print(sampled_df.head())

['image', 'ID', 'Subset', 'Question', 'Translated Question', 'Options', 'Translated Options', 'Label', 'Category', 'Image Type', 'Image Source', 'License', 'language', 'country']
                                               image                     ID  \
0  {'bytes': b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x...  5865745404279595472_2   
1  {'bytes': b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x...  5865745404276766066_2   
2  {'bytes': b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x...  5865745414278902903_1   
3  {'bytes': b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x...  5865745404277750076_0   
4  {'bytes': b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x...  5865745414271071540_1   

                         Subset  \
0  ('Egyptian_Arabic', 'Egypt')   
1  ('Egyptian_Arabic', 'Egypt')   
2  ('Egyptian_Arabic', 'Egypt')   
3  ('Egyptian_Arabic', 'Egypt')   
4  ('Egyptian_Arabic', 'Egypt')   

                                            Question  \
0  في انهي مدينة حققت اللاعبة المصرية ميداليتها ا...   
1                   ايه

In [4]:
EMBEDDING_SIZE = 1024
cvqa_index_body = {
    "settings":{
        "index":{
            "knn": True,
            "number_of_shards": 1,
            "number_of_replicas": 0
        },
        "analysis": {
            "analyzer":{
                "multilingual_analyzer":{ #if we want to use BM25 retrieval
                    "type":"standard",
                    "stopwords": "_none_"
                }
            }
        }
    },
    "mappings":{
        "properties":{
            "question_id": {"type": "keyword"},
            "question": {"type": "text", "analyzer": "multilingual_analyzer"},
            "translated_question":{"type": "text", "analyzer": "multilingual_analyzer"},
            "question_vector":{ #sbert embedding
                "type": "knn_vector",
                "dimension": EMBEDDING_SIZE,
                "method":{
                    "name": "hnsw",
                    "space_type":"cosinesimil",
                    "engine":"faiss"
                }
            },
            "options": {"type": "keyword"}, #possible answers
            "translated_options": {"type": "keyword"},
            "language": {"type": "keyword"},
            "country": {"type": "keyword"},
            "category": {"type": "keyword"},
            #"image_id": {"type": "keyword"},
            #"image_caption": {"type": "text", "analyzer": "multilingual_analyzer"},
            "image_source": {"type": "keyword", "index": False},
            "agent_answer_baseline": {"type": "keyword"},
            "agent_answer_augmented": {"type": "keyword"},
            "baseline_correct": {"type": "boolean"},
            "augmented_correct": {"type": "boolean"}
        }
    }
}

wiki_cache_index_body = {
    "settings":{
        "index":{
            "knn":True,
            "number_of_shards":1,
            "number_of_replicas":0
        },
        "analysis": {
            "analyzer":{
                "multilingual_analyzer":{
                    "type":"standard",
                    "stopwords": "_none_"
                }
            }
        }
    },
    "mappings":{
        "properties":{
            "doc_id": {"type": "keyword"}, #hash of title+chunk: sha256(title+chunk)
            "title": {"type": "text", "analyzer": "multilingual_analyzer"},
            "passage": {"type": "text", "analyzer": "multilingual_analyzer"},
            "passage_vector":{ #sbert embedding
                "type": "knn_vector",
                "dimension": EMBEDDING_SIZE,
                "method":{
                    "name": "hnsw",
                    "space_type":"cosinesimil",
                    "engine":"faiss"
                }
            },
            "language": {"type": "keyword"},
            "wikipedia_url": {"type": "keyword", "index": False},
            "wikipedia_title":{"type": "keyword"},
            "chunk_index": {"type": "integer"},
            "retrieved_for_question_ids": {"type": "keyword"},
            "retrieval_count": {"type": "integer"}
        }
    }
}

In [5]:

for index_name in [cvqa_index_name, wiki_cache_index, OPENSEARCH_USER + '_project']:

    exists = client.indices.exists(index=index_name)
    print(f"Index '{index_name}' exists: {exists}")
    # if(exists):
    #     client.indices.close(index=index_name)
    #     print(f"Index '{index_name}' closed successfully.")
#client.close()

Index 'uservl07_cvqa_project' exists: True
Index 'uservl07_wiki_cache' exists: True
Index 'uservl07_project' exists: True


In [6]:
if not client.indices.exists(index=cvqa_index_name):
    client.indices.create(index=cvqa_index_name, body=cvqa_index_body)
    print(f"Created OpenSearch index: {cvqa_index_name}")
else:
    print(f"OpenSearch index already exists: {cvqa_index_name}")
if not client.indices.exists(index=wiki_cache_index):
    client.indices.create(index=wiki_cache_index, body=wiki_cache_index_body)
    print(f"Created OpenSearch index: {wiki_cache_index}")
else:
    print(f"OpenSearch index already exists: {wiki_cache_index}")

OpenSearch index already exists: uservl07_cvqa_project
OpenSearch index already exists: uservl07_wiki_cache


In [7]:
#compute embeddings and save them locally
from pathlib import Path
import json

BATCH_SIZE = 64
CHECKPOINT_DIR = Path("embedding_chkpt")
CHECKPOINT_DIR.mkdir(exist_ok=True)
EMBEDDING_PATH = Path("cvqa_question_embeddings.npz")
METADATA_PATH = Path("cvqa_retrieval_metadata.json")

#print(retrieval_corpus.column_names)
all_ids = [row['ID'] for row in retrieval_corpus]
all_texts = [row['Question'] for row in retrieval_corpus]

if not EMBEDDING_PATH.exists():
    metadata = json.dump({
        "model": bge_model.__class__.__name__,
        "embedding_size": EMBEDDING_SIZE,
        "prefix_query": None,
        "prefix_passage": None,
        "normalized": True,
        "dataset": "afaji/cvqa",
        "num_items": len(all_texts),
    }, open(METADATA_PATH, "w"))

if not EMBEDDING_PATH.exists():
    for i in range(0, len(all_texts), BATCH_SIZE):
        checkpoint_path = CHECKPOINT_DIR / f"batch_{i}.npz"

        if checkpoint_path.exists():
            print(f"Batch {i} already processed, skipping.")
            continue
        batch_ids = all_ids[i:i+BATCH_SIZE]
        batch_texts = all_texts[i:i+BATCH_SIZE]

        batch_vectors= bge_model.encode(
            batch_texts,
            normalize_embeddings=True,
            show_progress_bar=False,
            batch_size=BATCH_SIZE
        )

        np.savez(checkpoint_path, ids=batch_ids, vectors=batch_vectors)
        print(f"Saved batch {i} / {len(all_texts)} to checkpoint.")
if not EMBEDDING_PATH.exists():
    all_npz_ids=[]
    all_npz_vectors=[]

    for path in sorted(CHECKPOINT_DIR.glob("batch_*.npz")):
        data = np.load(path)
        all_npz_ids.extend(data['ids'])
        all_npz_vectors.extend(data['vectors'])

    np.savez(
        EMBEDDING_PATH,
        ids=np.array(all_npz_ids),
        vectors=np.array(all_npz_vectors)
    )

    print(f"All embeddings computed and saved to {EMBEDDING_PATH}")
else:
    print(f"Embeddings file {EMBEDDING_PATH} already exists, skipping computation.")

Embeddings file cvqa_question_embeddings.npz already exists, skipping computation.


In [ ]:
#TODO: Recreate index with the correct documents

def generate_docs(dataset, embedding_path):
    data = np.load(embedding_path)
    id_to_vector =dict(zip(data['ids'], data['vectors']))

    for row in dataset:
        qid = row['ID']
        yield{
            "_index": cvqa_index_name,
            "_id": qid,
            "_source":{
                "question_id": qid,
                "question": row['Question'],
                "translated_question": row['Translated Question'],

                "question_vector": id_to_vector[qid].tolist(),
                "options": row['Options'],
                "translated_options": row['Translated Options'],
                "language": row['language'],
                "country": row['country'],
                "category": row['Category'],
                # "image_id": str(row['image']),
                # "image_caption": row.get('image_caption', None),
                "image_source": row['Image Source'],
                # "agent_answer_baseline": None,
                # "agent_answer_augmented": None,
                # "baseline_correct": None,
                # "augmented_correct": None
            }
        }


client.indices.open(index=cvqa_index_name)
count = client.count(index=cvqa_index_name)['count']
if count == 0:
   print(f"Indexing documents into OpenSearch index: {cvqa_index_name}...")
   helpers.bulk(
    client,
    generate_docs(retrieval_corpus, EMBEDDING_PATH),
    chunk_size=200,
    request_timeout=60
    )
else:
    print(f"Index {cvqa_index_name} already has {count} documents, skipping indexing.")
print(f"Total documents indexed in {cvqa_index_name}: {count}")
if count != len(retrieval_corpus):
    print("Warning: Document count in OpenSearch does not match expected count from dataset.")
else:   
    print("Indexed all documents into OpenSearch successfully.")

Index uservl07_cvqa_project already has 227 documents, skipping indexing.
Total documents indexed in uservl07_cvqa_project: 227
Indexed all documents into OpenSearch successfully.


3. Baseline evaluation: Evaluate Gemma 4 directly on the test set questions
without retrieval. Report accuracy per language and cultural group. Identify
which groups and question types show the largest failure rates.



In [44]:
from openai import OpenAI
import base64
client = OpenAI(base_url=BASE_URL, api_key=API_KEY)

def ask_gemma(client:OpenAI, question:str,options:list[str], image:Image) -> str:
    if isinstance(image, dict):
        image = Image.open(io.BytesIO(image['bytes'])).convert("RGB")
    buffered = io.BytesIO()
    image.save(buffered, format="JPEG")
    image_b64 = base64.b64encode(buffered.getvalue()).decode()
    QUESTION_PROMPT=f"""
    Question: {question}
    Options:
    0: {options[0]}
    1: {options[1]}
    2: {options[2]}
    3: {options[3]}
    """
    content = [
        {"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{image_b64}"}},
        {"type": "text", "text": QUESTION_PROMPT}
    ]
    PROMPT="""
    You are a helpful visual question answering assistant. 
    You will be given an image, a question, and four options.
    Respond with only the number of the correct option: 0,1,2, or 3.
    Do not explain your answer. Do not write anything else.    
    """
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": PROMPT},
            {"role": "user", "content": content}
        ]
    )
    return response.choices[0].message.content


In [45]:
#evaluation generates a free-form answer compared to the options through embeddings
def evaluate(response,label):
    response = response.strip()
    label = str(label)
    if response in ['0','1','2','3']:
        return {
            "predicted_label": response,
            "correct": label,
            "is_correct": response == label
        }
    else:
        return {
            "predicted_label": None,
            "correct": label,
            "is_correct": False
        }

In [46]:
for i in range(2):
    test_item = test_set[i]
    print("Question:", test_item['Question'])
    print("Options:", test_item['Options'])
    print("Translated Question:", test_item['Translated Question'])
    print("Translated Options:", test_item['Translated Options'])
    print("Label:", test_item['Label'])
    response = ask_gemma(openai_client, test_item['Question'], test_item['Options'], test_item['image'])
    print("Gemma Response:", response)
    eval_result = evaluate(response, test_item['Label'])
    print("Evaluation Result:", eval_result)


Question: Em que lugar está sendo realizada a atividade mostrada na foto?
Options: ['Em um clube', 'Numa quadra de escola de samba', 'Em um ginásio esportivo', 'Num celeiro']
Translated Question: Where is the activity in the picture taking place?
Translated Options: ['In a club', 'At a samba school court', 'In a sports gym', 'In a barn']
Label: 1
Gemma Response: 1
Evaluation Result: {'predicted_label': '1', 'correct': '1', 'is_correct': True}
Question: في أنهي سنة إتوفى؟
Options: ['٢٠٢٠', '٢٠١٨', '٢٠١٦', '٢٠١٤']
Translated Question: In which year did he die?
Translated Options: ['2020', '2018', '2016', '2014']
Label: 2
Gemma Response: 1
Evaluation Result: {'predicted_label': '1', 'correct': '2', 'is_correct': False}


In [47]:
def evaluate_dataset(dataset, save_path):
    results = []
    for row in dataset:
        response = ask_gemma(openai_client, row['Question'], row['Options'], row['image'])
        eval_result = evaluate(response, row['Label'])
        results.append({
            "question_id": row['ID'],
            "language": row['language'],
            "country": row['country'],
            "category": row['Category'],
            "question": row['Question'],
            "options": row['Options'],
            "label": row['Label'],
            "predicted_label": eval_result['predicted_label'],
            "is_correct": eval_result['is_correct'],
            "is_None": eval_result['predicted_label'] is None
        })
    results_df = pd.DataFrame(results)
    results_df.to_csv(save_path, index=False)
    print("Evaluation complete. Results saved to", save_path)

In [48]:
evaluate_dataset(test_set, "cvqa_gemma_evaluation_results.csv")
results_df = pd.read_csv("cvqa_gemma_evaluation_results.csv")
accuracy = results_df['is_correct'].mean()
print(f"Overall Accuracy on CVQA Test Set: {accuracy:.4f}")
null_rate = results_df['is_None'].mean()
print(f"Rate of None predictions: {null_rate:.4f}")

#per language accuracy
language_accuracy = results_df.groupby('language')['is_correct'].mean()
print("\nAccuracy by Language:")
for lang, acc in language_accuracy.items():
    print(f"  {lang}: {acc:.4f}")
#country accuracy
country_accuracy = results_df.groupby('country')['is_correct'].mean()
print("\nAccuracy by Country:")
for country, acc in country_accuracy.items():
    print(f"  {country}: {acc:.4f}")
#category accuracy
category_accuracy = results_df.groupby('category')['is_correct'].mean()
print("\nAccuracy by Category:")
for category, acc in category_accuracy.items():
    print(f"  {category}: {acc:.4f}")


Evaluation complete. Results saved to cvqa_gemma_evaluation_results.csv
Overall Accuracy on CVQA Test Set: 0.8265
Rate of None predictions: 0.0000

Accuracy by Language:
  0: 0.7805
  1: 0.8596

Accuracy by Country:
  Brazil: 0.8596
  Egypt: 0.7805

Accuracy by Category:
  Brands / products / companies: 0.6667
  Cooking and food: 0.8750
  Geography / buildings / landmarks: 0.9200
  Objects / materials / clothing: 0.8571
  People and everyday life: 1.0000
  Plants and animal: 1.0000
  Public Figure and pop culture: 0.5000
  Sports and recreation: 1.0000
  Traditions / art / history: 0.7273
  Vehicles and Transportation: 0.7500


4. Retrieval-augmented cultural reasoning: Extend the agent with two
complementary tools: (a) a retrieve_similar_questions(query,
language) tool that retrieves related CVQA image–question pairs from the
retrieval index, and (b) a WikipediaSearchTool (built into smolagents) for
on-demand cultural background — retrieved Wikipedia articles are cached
into the OpenSearch index as they are fetched, so the knowledge base
grows incrementally. Pass the retrieved context to the LVL
M alongside the
test set question and measure the accuracy improvement.

5. Gap analysis: Compare performance across your three cultural groups.
Characterise the failure modes — are errors due to visual recognition,
cultural knowledge, or language understanding?

In [ ]:
for index in [cvqa_index_name, wiki_cache_index, OPENSEARCH_USER + '_project']:
    if client.indices.exists(index=index):
        client.indices.close(index=index)
        print(f"Index '{index}' closed successfully.")

client.close()

Index 'uservl07_cvqa_project' closed successfully.
Index 'uservl07_wiki_cache' closed successfully.
Index 'uservl07_project' closed successfully.
